In [ ]:

from modelscope import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
prompt = "你是谁"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
print(text)


In [ ]:
model_inputs = tokenizer([text], return_tensors="pt")
generated_ids = model.generate(**model_inputs, max_new_tokens=32768)
content = tokenizer.decode(generated_ids[0])
print(content)


In [ ]:
data = [
    {"Q": "你是谁", "A": "我是你的学习助手"},
    {"Q": "你是谁开发的", "A": "我是由慧科团队开发的"},
    {"Q": "你能做什么", "A": "我能为你解答学习问题"},
]

In [ ]:


from torch.utils.data import Dataset, DataLoader

lora_prompt_templates = """
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
<think>

</think>
{answer}<|im_end|>
"""


class dataset(Dataset):
    def __init__(self, data, max_length=128):
        self.encodings = []
        for qa in data:
            text = lora_prompt_templates.format(question=qa['Q'], answer=qa['A'])
            encoded = tokenizer(text, max_length=max_length, truncation=True, padding='max_length',
                                return_tensors='pt')
            inpus_ids = encoded['input_ids'].squeeze()
            self.encodings.append(inpus_ids)

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        return self.encodings[idx]


train_dataset = dataset(data)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
for batch in train_loader:
    print(batch)
    break


In [ ]:
print(model)

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=2,
    target_modules=["q_proj", "v_proj", "k_proj"]
)
lora_model = get_peft_model(model, lora_config)

lora_model.print_trainable_parameters()


In [ ]:
print(model)

In [ ]:
import torch

optimizer = torch.optim.Adam(lora_model.parameters(), lr=0.0001)

step = 10
for epoch in range(step):
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = lora_model(batch, labels=batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
    print(loss.item())

In [ ]:
prompt = "你是谁"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
model_inputs = tokenizer([text], return_tensors="pt")
generated_ids = lora_model.generate(**model_inputs, max_new_tokens=32768)
content = tokenizer.decode(generated_ids[0])
print(content)


In [ ]:
lora_model.save_pretrained("../model/loraQwen3")


In [1]:

from modelscope import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_name = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = PeftModel.from_pretrained(model, "../model/loraQwen3")


C:\Users\17246\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2025-08-12 10:51:17,094 - modelscope - INFO - Creating symbolic link [C:\Users\17246\.cache\modelscope\hub\models\Qwen\Qwen3-0.6B].
2025-08-12 10:51:17,096 - modelscope - WARNING - Failed to create symbolic link C:\Users\17246\.cache\modelscope\hub\models\Qwen\Qwen3-0.6B for C:\Users\17246\.cache\modelscope\hub\models\Qwen\Qwen3-0___6B.


2025-08-12 10:51:20,056 - modelscope - INFO - Creating symbolic link [C:\Users\17246\.cache\modelscope\hub\models\Qwen\Qwen3-0.6B].
2025-08-12 10:51:20,057 - modelscope - WARNING - Failed to create symbolic link C:\Users\17246\.cache\modelscope\hub\models\Qwen\Qwen3-0.6B for C:\Users\17246\.cache\modelscope\hub\models\Qwen\Qwen3-0___6B.


In [4]:
prompt = "你能做什么"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
model_inputs = tokenizer([text], return_tensors="pt")
generated_ids = model.generate(**model_inputs, max_new_tokens=32768)
content = tokenizer.decode(generated_ids[0])
print(content)

<|im_start|>user
你能做什么<|im_end|>
<|im_start|>assistant
<think>

</think>

我能为你解答学习问题<|im_end|>
